# B2 · Regression trees on house prices

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Compare a linear regression with a regression tree on the California prices, look at how the validation RMSE changes as `max_depth` and `min_samples_leaf` vary, choose both with a search, and check how much the geographic position is worth (a tree on latitude and
longitude alone).

## Dataset

`ds.california()` (20,640 districts, 8 features; target: median price in hundreds of thousands of dollars), 75/25 split with `random_state=0`; RMSE with `squared=False`. Validation: `KFold(5, shuffle=True, random_state=0)` on the training set.

## Educational purpose

See that an **unregularized** tree does not beat linear regression, that with the right minimum leaf size it beats it by a lot, and compare with the geographic KNN of module 5 (0.53): the proximity structure mattered a lot here too.

## Task

1. Save `rmse` (dictionary) on the test set of `linear` (`LinearRegression`) and `base tree` (`DecisionTreeRegressor(random_state=0)` with no parameters).
2. With `KFold(5, shuffle=True, random_state=0)` on the training set compute `depth_curve` (`pd.Series`, mean RMSE for `max_depth` in `[3, 5, 8, 12, 50]`: 50 is equivalent to no limit) and `leaf_curve` (mean RMSE for `min_samples_leaf` in `[1, 5, 20, 50, 100]`, no depth limit). Save `best_depth` and `best_leaf`.
3. `GridSearchCV` (same `cv`, `scoring="neg_root_mean_squared_error"`) on `max_depth` in `[6, 8, 10, 12, None]` and `min_samples_leaf` in `[5, 10, 20, 40]`: save `params` (dictionary), `tree_rmse` (test RMSE) and `n_leaves`.
4. Train a tree with only the pair `Latitude`, `Longitude` and `min_samples_leaf=20`: save `geo_rmse`. Answer with `geo_is_enough = True` if `geo_rmse` is within 0.05 of `tree_rmse`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
cv = KFold(5, shuffle=True, random_state=0)
error = lambda model, columns=None: mean_squared_error(y_te, model.fit(X_tr if columns is None else X_tr[columns], y_tr).predict(X_te if columns is None else X_te[columns]), squared=False)

## Your work

In [ ]:
rmse = {}

In [ ]:
depth_curve = pd.Series(dtype=float)
leaf_curve = pd.Series(dtype=float)
best_depth = None
best_leaf = None

In [ ]:
params = None
tree_rmse = None
n_leaves = None

In [ ]:
geo_rmse = None
geo_is_enough = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`cross_val_score(DecisionTreeRegressor(max_depth=d, random_state=0), X_tr, y_tr, cv=cv, scoring="neg_root_mean_squared_error")`, then `-mean`. Use a dictionary: `pd.Series({d: cv_rmse(max_depth=d) for d in [3, 5, 8, 12, 50]})`.'):
    check(set(rmse) == {"linear", "base tree"} and abs(rmse["base tree"] - rmse["linear"]) < 0.03, "the unregularized tree (0.74) is on par with linear regression (0.735)", "error(model) with squared=False")
    check(list(depth_curve.index) == [3, 5, 8, 12, 50] and best_depth == 8 and depth_curve[8] < depth_curve[50] - 0.05, "the best depth is 8; with no limit (50) the RMSE is over 0.05 higher", "idxmin of the curve")
    check(list(leaf_curve.index) == [1, 5, 20, 50, 100] and best_leaf in (20, 50) and leaf_curve[1] > leaf_curve[best_leaf] + 0.1, "best min_samples_leaf 20 or 50; with 1 the RMSE is over 0.1 higher", "idxmin of the curve")
    check(params["min_samples_leaf"] in (10, 20, 40) and tree_rmse < 0.65 and tree_rmse < rmse["linear"] - 0.08, "the optimized tree beats the linear one by over 0.08 (RMSE below 0.65)", "GridSearchCV on max_depth and min_samples_leaf")
    check(n_leaves >= 200, "the best tree has hundreds of leaves: unlike the Titanic many rules are needed here", "get_n_leaves() of the final model")
    check(geo_rmse < 0.66 and geo_is_enough is True, "with the geographic position alone the RMSE is within 0.05 of the full tree", "DecisionTreeRegressor on the two coordinates only")
done()

---
The commented solution is in `solutions/B_with_sklearn/B2_regression_tree_on_california.ipynb`. Open it after you pass the check, or after you have really tried.